# SANAD RC3 — LensFD FULL DATA: Direct Source Recovery v3

**Why:** Two 18,898,485,248-byte downloads of the publisher TAR failed GZIP/TAR CRC and end-of-file tests. This notebook **does not download the broken `.tgz` again**. It first enumerates the publisher's `data_LensFD/` directory and independently downloads **every published regular data file**, scanning all numeric HDF5 values. Prior `members.jsonl` stays intact; validated files are reused with explicit provenance.

**Important:** The publisher's nested directory index and direct-file permissions are checked at runtime. If they are unavailable, the notebook refuses to pretend the dataset is complete. **No sampling or skipping.** All raw measurements stay in private Drive/local runtime. This does not change frozen RC3.


In [ ]:
%pip -q install 'numpy>=1.26,<3' 'h5py>=3.11,<5' 'requests>=2.31,<3' 'pytest>=8,<10'
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, shutil, subprocess, sys
ROOT=Path('/content/drive/MyDrive/SANAD_Network/RC3_LensFD_FULL')
OLD_RESULTS=ROOT/'results_full'
RESULTS=ROOT/'results_direct'
RESULTS.mkdir(parents=True,exist_ok=True)
SCRATCH=Path('/content')
OLD_JOURNAL=OLD_RESULTS/'members.jsonl'
REVALIDATE_OLD_FILES=False  # True = re-download and compare every previously audited file to official direct source
print('Previous completed-member journal:',OLD_JOURNAL.exists())
print('Scratch free GiB:',round(shutil.disk_usage(SCRATCH).free/2**30,2))
print('Do not delete previous archive journal. No damaged 18GB tar downloads in v3.')


In [ ]:
from pathlib import Path
Path('/content/lensfd_full_run.py').write_text('"""Full-archive LensFD source audit and frozen RC3 compatibility gate.\n\nRead every byte of every regular archive member and every element in every numeric\nHDF5 dataset. No sampling, no silent truncation, no fabricated RC3 features.\nSource: Rice University LensFD (Chen et al., IEEE TVT 2023).\nLicense: Rice grants non-exclusive/non-transferable use; raw data must not be redistributed.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport subprocess\nfrom datetime import datetime, timezone\nfrom itertools import product\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Iterator\n\nimport numpy as np\n\nBLOCK_BYTES = 16 * 1024 * 1024\nOFFICIAL_URL = \'https://renew.rice.edu/datasets/ashu/traces/LensFD/data_lensFD.tgz\'\nEXPECTED_FROZEN_COMMIT = \'182e77b485cceb770ee6570ba332205c73823c91\'\nEXPECTED_MODEL_SHA256 = \'0a90835bb57a56437fc09211902b581bb22377676f91dce68f309c92e3a95bf4\'\n\n\ndef utcnow():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef hash_file(p: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:\n    h = hashlib.sha256()\n    with Path(p).open(\'rb\') as file:\n        for b in iter(lambda: file.read(chunk_bytes), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef atomic_json(path: Path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + \'.tmp\')\n    with temp.open(\'w\', encoding=\'utf8\') as f:\n        json.dump(obj, f, indent=2, ensure_ascii=False, allow_nan=False)\n        f.flush(); os.fsync(f.fileno())\n    os.replace(temp, path)\n\n\ndef _json_line(path: Path, obj):\n    with Path(path).open(\'a\', encoding=\'utf8\') as f:\n        f.write(json.dumps(obj, ensure_ascii=False, allow_nan=False) + \'\\n\')\n        f.flush(); os.fsync(f.fileno())\n\n\ndef iter_dataset_slices(shape, itemsize, target_bytes=BLOCK_BYTES) -> Iterator[tuple]:\n    """Cover the complete N-D logical HDF5 space exactly once, at bounded memory."""\n    if not shape:\n        yield ()\n        return\n    blocks = list(map(int, shape))\n    if any(k <= 0 for k in blocks):\n        return\n    while math.prod(blocks) * max(1, itemsize) > target_bytes:\n        largest = max(range(len(blocks)), key=lambda i: blocks[i])\n        if blocks[largest] == 1:\n            raise MemoryError(\'Single HDF5 element larger than target block\')\n        blocks[largest] = (blocks[largest] + 1) // 2\n    ranges = [range(0, n, w) for n,w in zip(shape, blocks)]\n    for start in product(*ranges):\n        yield tuple(slice(s, min(s+b,n)) for s,b,n in zip(start,blocks,shape))\n\n\ndef _scan_numeric(ds) -> dict:\n    dtype = ds.dtype\n    count = int(math.prod(ds.shape)) if ds.shape else 1\n    record = dict(shape=list(ds.shape), dtype=str(dtype), values=count)\n    if dtype.kind not in (\'b\',\'i\',\'u\',\'f\',\'c\'):\n        record.update(numeric_analyzed=False, values_scanned=0, reason=\'Non-plain-numeric HDF5 dtype; complete source bytes still SHA-256 verified\')\n        return record\n    total = 0\n    zero = 0\n    finite = 0\n    abs_sum = 0.0\n    sq_sum = 0.0\n    mn = float(\'inf\'); mx = float(\'-inf\')\n    for sl in iter_dataset_slices(ds.shape, int(dtype.itemsize)):\n        vals = np.asarray(ds[sl] if sl else ds[()]).reshape(-1)\n        total += len(vals)\n        valid = np.isfinite(vals)\n        finite += int(np.count_nonzero(valid))\n        good = vals[valid]\n        if good.size:\n            mag = np.abs(good.astype(np.complex128 if np.iscomplexobj(good) else np.float64))\n            zero += int(np.count_nonzero(mag == 0))\n            abs_sum += float(np.sum(mag, dtype=np.float64))\n            sq_sum += float(np.sum(mag * mag, dtype=np.float64))\n            mn = min(mn, float(np.min(mag)))\n            mx = max(mx, float(np.max(mag)))\n    if total != count:\n        raise RuntimeError(f\'Numeric coverage incomplete {ds.name}: {total}/{count}\')\n    record.update(numeric_analyzed=True,values_scanned=total,finite_values=finite,\n                  nonfinite_values=total-finite,zero_values=zero,\n                  mean_abs=abs_sum/finite if finite else None,\n                  rms_abs=math.sqrt(sq_sum/finite) if finite else None,\n                  min_abs=mn if finite else None,max_abs=mx if finite else None)\n    return record\n\n\ndef inspect_hdf5(path: Path):\n    import h5py\n    rows=[]; n_numeric=0; n_analyzed=0; total_values=0\n    with h5py.File(path, \'r\') as f:\n        def visit(name, item):\n            nonlocal n_numeric,n_analyzed,total_values\n            if not isinstance(item,h5py.Dataset):\n                return\n            stats=_scan_numeric(item)\n            stats[\'dataset_path\']=\'/\'+name\n            stats[\'attribute_names\']=sorted(list(item.attrs.keys()))\n            rows.append(stats)\n            if stats[\'numeric_analyzed\']:\n                n_numeric+=1\n                n_analyzed+=stats[\'values_scanned\']\n            total_values+=stats[\'values\']\n        f.visititems(visit)\n        root_attrs=sorted(list(f.attrs.keys()))\n    return {\'hdf5_datasets\':rows,\'hdf5_dataset_count\':len(rows),\n            \'numeric_dataset_count\':n_numeric,\'numeric_values_scanned\':n_analyzed,\n            \'total_logical_values\':total_values,\'root_attribute_names\':root_attrs}\n\n\ndef _safe_member_name(name):\n    p=Path(name)\n    if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in name or not name or name.startswith(\'~\'):\n        raise ValueError(\'Unsafe archive member path\')\n    return str(p)\n\n\ndef _load_journal(journal):\n    if not journal.exists():return {}\n    records={}\n    with journal.open(encoding=\'utf8\') as f:\n        for line in f:\n            if line.strip():\n                row=json.loads(line)\n                # Append-only retry journal; most recent attempt for each member wins.\n                records[row[\'name\']]=row\n    return records\n\n\ndef validate_archive_structure(archive: str | Path) -> dict:\n    """Exhaustively validate gzip CRC and TAR member framing without extracting data.\n\n    GNU tar streams through all bytes, so this checks the large final member too.\n    Merely hashing an incomplete file does NOT prove completeness.\n    """\n    archive = Path(archive)\n    check = subprocess.run([\'tar\', \'-tzf\', str(archive)], stdout=subprocess.DEVNULL,\n                           stderr=subprocess.PIPE, text=True, errors=\'replace\')\n    result = {\'archive\': str(archive), \'archive_size\': archive.stat().st_size,\n              \'tar_exit_code\': check.returncode,\n              \'complete\': check.returncode == 0,\n              \'stderr_tail\': check.stderr[-2400:]}\n    return result\n\n\ndef archive_preflight(archive, output, *, require_complete=True):\n    """Prevent a slow HDF5 scan from ending on a truncated archive."""\n    output = Path(output); output.mkdir(parents=True,exist_ok=True)\n    report = validate_archive_structure(archive)\n    atomic_json(output/\'ARCHIVE_PREFLIGHT.json\', report)\n    if not report[\'complete\'] and require_complete:\n        raise RuntimeError(\'LensFD TAR/GZIP integrity FAILED before HDF5 scanning. \'\n                           \'Inspect ARCHIVE_PREFLIGHT.json; repair/re-download the \'\n                           \'source archive, then rerun. Existing members.jsonl \'\n                           \'and completed HDF5 statistics are preserved. \'\n                           + report[\'stderr_tail\'][-500:])\n    return report\n\n\ndef _archive_summary(*, archive, arch_sha, seen, prior, dirs, errors, source_changed):\n    regular = [r for r in prior.values() if r[\'name\'] in seen]\n    ok = [r for r in regular if r[\'status\'] == \'OK\']\n    complete = (len(regular) == len(seen) and len(ok) == len(seen) and not errors)\n    source_groups = Counter(Path(r[\'name\']).name.split(\'-\')[0].lower() for r in ok)\n    return {\'status\':\'COMPLETE_FULL_ARCHIVE_AUDIT\' if complete else \'INCOMPLETE\',\n       \'official_source\':OFFICIAL_URL, \'finished_utc\':utcnow(),\n       \'archive_sha256\':arch_sha, \'archive_bytes_hashed\':archive.stat().st_size,\n       \'archive_members_regular\':len(seen), \'archive_members_directories\':dirs,\n       \'source_filename_prefix_counts\':dict(sorted(source_groups.items())),\n       \'members_checked\':len(regular), \'members_ok\':len(ok),\n       \'member_bytes_accounted\':sum(r[\'size\'] for r in ok),\n       \'hdf5_dataset_count\':sum(r.get(\'hdf5_dataset_count\',0) for r in ok),\n       \'hdf5_numeric_values_scanned\':sum(r.get(\'numeric_values_scanned\',0) for r in ok),\n       \'hdf5_total_logical_values\':sum(r.get(\'total_logical_values\',0) for r in ok),\n       \'hdf5_nonnumeric_dataset_count\':sum(r.get(\'hdf5_dataset_count\',0)-r.get(\'numeric_dataset_count\',0) for r in ok),\n       \'bytes_sampled\':0, \'inference_source\':\'RAW_REAL_RF_MEASUREMENTS\',\n       \'rc3_qualification_on_lensfd\':\'NOT_YET_TESTED\',\n       \'archive_replacement_detected\':bool(source_changed),\n       \'errors\':errors + [{\'name\':r[\'name\'],\'error\':r.get(\'error\',\'unresolved\')}\n                          for r in regular if r[\'status\']!=\'OK\' and r[\'name\'] not in {e[\'name\'] for e in errors}]}\n\n\ndef run_full_archive(archive: str | Path, output: str | Path, scratch_dir: str | Path | None = None,\n                     strict=True, *, preflight=True) -> dict:\n    """Audit every regular TAR member; preserve checkpoints, including on TAR read errors.\n\n    Reused source: completed HDF5 member checks are not repeated.\n    Replaced source: re-hash cached members BEFORE reusing their HDF5 statistics.\n    No sample limits, no silent skipping.  The frozen RC3 model is not changed.\n    """\n    archive = Path(archive); output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    if not archive.is_file(): raise FileNotFoundError(str(archive))\n    if preflight: archive_preflight(archive, output)\n    arch_size=archive.stat().st_size\n    arch_sha=hash_file(archive)\n    lock=output/\'SOURCE_LOCK.json\'\n    previous=json.loads(lock.read_text()) if lock.exists() else None\n    source_changed=bool(previous and\n        (previous[\'archive_sha256\'],previous[\'archive_size\'])!=(arch_sha,arch_size))\n    # Do NOT discard old journal simply because source was repaired; verify each\n    # cached member\'s SHA against the replacement archive before trusting stats.\n    journal=output/\'members.jsonl\'; prior=_load_journal(journal)\n    seen=set();errors=[];dirs=0;tar_error=None\n    scratch=Path(scratch_dir or tempfile.gettempdir()); scratch.mkdir(parents=True,exist_ok=True)\n    try:\n        with tarfile.open(archive,\'r|gz\') as tar:\n            for member in tar:\n                name=_safe_member_name(member.name)\n                if member.isdir(): dirs+=1; continue\n                if name in seen: raise RuntimeError(f\'Duplicate archive member {name}\')\n                seen.add(name)\n                old=prior.get(name)\n                if old and old[\'status\']==\'OK\' and old[\'size\']==member.size and not source_changed:\n                    continue\n                row={\'name\':name, \'size\':member.size,\'mtime\':member.mtime,\'status\':\'ERROR\',\n                     \'archive_sha256\':arch_sha}\n                tempname=None\n                try:\n                    if not member.isfile():\n                        raise ValueError(f\'Unsupported non-regular archive object: {member.type!r}\')\n                    inp=tar.extractfile(member)\n                    if inp is None:raise RuntimeError(\'Could not access archive member stream\')\n                    # If source archive was repaired, verify old 112+ completed members\n                    # by hashing them directly without extracting / re-scanning HDF5.\n                    if old and old[\'status\']==\'OK\' and old[\'size\']==member.size and source_changed:\n                        dig=hashlib.sha256(); remaining=member.size\n                        while remaining:\n                            b=inp.read(min(8*1024*1024,remaining))\n                            if not b: raise EOFError(f\'Truncated archive member {name}\')\n                            dig.update(b); remaining-=len(b)\n                        if dig.hexdigest()!=old.get(\'sha256\'):\n                            raise RuntimeError(\'PREVIOUS_OK_FILE_CHANGED; cannot reuse its HDF5 statistics. \'\n                                               \'Use a fresh audit folder for this different source.\')\n                        row={**old,\'archive_sha256\':arch_sha,\n                             \'cache_verified_against_repaired_archive\':True}\n                    else:\n                        free=shutil.disk_usage(str(scratch)).free\n                        margin=256*1024*1024\n                        if member.size+margin>free:\n                            raise OSError(\'INSUFFICIENT_SCRATCH_DISK: \'\n                               f\'{name} is {member.size:,} bytes ({member.size/2**30:.2f} GiB); \'\n                               f\'/content has only {free:,} bytes ({free/2**30:.2f} GiB) free; \'\n                               \'select a larger-disk Colab runtime. NO SUBSAMPLING.\')\n                        dig=hashlib.sha256()\n                        with tempfile.NamedTemporaryFile(prefix=\'lensfd_\',\n                            suffix=Path(name).suffix,dir=str(scratch),delete=False) as temp:\n                            tempname=temp.name; remaining=member.size\n                            while remaining:\n                                b=inp.read(min(8*1024*1024,remaining))\n                                if not b: raise EOFError(f\'Truncated archive member {name}\')\n                                temp.write(b);dig.update(b);remaining-=len(b)\n                        row[\'sha256\']=dig.hexdigest()\n                        if Path(name).suffix.lower() in (\'.hdf5\',\'.h5\'):\n                            row.update(inspect_hdf5(Path(tempname)))\n                            row[\'analysis\']=\'FULL_HDF5_NUMERIC_SCAN\'\n                        else:\n                            row[\'analysis\']=\'FULL_BYTE_SHA256_ONLY\'\n                            row[\'warning\']=\'Non-HDF5 member fully SHA-256 hashed\'\n                        row[\'status\']=\'OK\'\n                except Exception as exc:\n                    row.update(status=\'ERROR\',error=f\'{type(exc).__name__}: {exc}\')\n                    errors.append({\'name\':name,\'error\':row[\'error\']})\n                finally:\n                    if tempname is not None:Path(tempname).unlink(missing_ok=True)\n                _json_line(journal,row); prior[name]=row\n                print(f\'[{len(seen)}] {row["status"]:5} {name}  bytes={member.size:,}\',flush=True)\n    except (tarfile.TarError, EOFError, OSError, RuntimeError, ValueError) as exc:\n        tar_error={\'name\':\'__ARCHIVE_STREAM__\',\'error\':f\'{type(exc).__name__}: {exc}\'}\n        errors.append(tar_error)\n    summary=_archive_summary(archive=archive, arch_sha=arch_sha, seen=seen,\n                             prior=prior,dirs=dirs,errors=errors,source_changed=source_changed)\n    if tar_error: summary[\'fatal_tar_stream_error\']=tar_error[\'error\']\n    atomic_json(output/\'FULL_AUDIT_SUMMARY.json\',summary)\n    if summary[\'status\']==\'COMPLETE_FULL_ARCHIVE_AUDIT\':\n        if source_changed and previous:\n            atomic_json(output/\'SOURCE_LOCK_PREVIOUS.json\',previous)\n        atomic_json(lock, {\'official_source\':OFFICIAL_URL,\'archive_size\':arch_size,\n                           \'archive_sha256\':arch_sha,\'verified_complete_utc\':utcnow()})\n    if strict and summary[\'status\']!=\'COMPLETE_FULL_ARCHIVE_AUDIT\':\n        detail=\'; \'.join(x[\'error\'] for x in summary[\'errors\'][-3:])\n        raise RuntimeError(\'Full LensFD archive audit INCOMPLETE. See \'\n                           \'FULL_AUDIT_SUMMARY.json and members.jsonl. \'+detail)\n    return summary\n\n\ndef rc3_contract_report(output_dir:str|Path, frozen_repository_path:str|Path|None=None):\n    """Fail closed: LensFD raw IQ is NOT the frozen RC3 300x7 observation trace.\n\n    RC3 requires real upstream evidence streams produced by the frozen Sionna\n    feature-generation pathway, not arbitrary tensor reshaping or synthetic rows.\n    """\n    reasons=[\n        \'RC3 input is an evidence NPZ of 300 reporting periods x 7 arrays with precomputed v11, fast/strict and temporal delta observations, not raw HDF5 IQ/CSI.\',\n        \'LensFD self-interference recording covers up to 32 antenna ports in any run; seven eight-element RC3 arrays must not be fabricated from asynchronous recordings.\',\n        \'LensFD source does not supply the frozen RC3 temporal feature stream, matching clock, 7-array baselines, or a machine-readable verified fault-label map by default.\',\n        \'Transforming raw IQ/CSI into plausible arrays or random RC3 observation features would not constitute an RC3 evaluation.\',\n    ]\n    out={\'dataset\':\'Rice LensFD\',\'mode\':\'FROZEN_RC3\',\'decision_threshold\':0.50,\n         \'expected_frozen_commit\':EXPECTED_FROZEN_COMMIT,\n         \'model_sha256\':EXPECTED_MODEL_SHA256,\'input_contract\':\'NPZ [2,300,7,3] plus [300,7,3] and v11 [300,7] per stream\',\n         \'adapter_status\':\'NO_VALIDATED_LENSFD_TO_RC3_FEATURE_ADAPTER\',\n         \'rc3_predictions_produced\':0,\'rc3_field_fault_detection_rate\':None,\n         \'reasons\':reasons,\'integrity_check_status\':\'SEPARATE_FROM_LENSFD_INFERENCE\'}\n    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)\n    atomic_json(output_dir/\'RC3_COMPATIBILITY.json\',out)\n    return out\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--archive\',required=True)\n    parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--scratch\',default=\'/content\')\n    args=parser.parse_args()\n    print(json.dumps(run_full_archive(args.archive,args.output,args.scratch),indent=2))\n    print(json.dumps(rc3_contract_report(args.output),indent=2))\n', encoding="utf-8")
print("Installed lensfd_full_run.py")

In [ ]:
from pathlib import Path
Path('/content/lensfd_direct_full.py').write_text('"""Full-source LensFD direct-directory recovery when publisher .tgz is truncated.\n\nNo partial-archive promotion, missing-record substitution, or sampling. Files\nare enumerated from the publisher\'s directory listing, downloaded individually,\nSHA-256 verified, and HDF5 numeric values scanned with bounded memory. Previously\ncompleted 112+ archive members may be reused with explicitly marked provenance.\n\nThe dataset is Rice University property; never redistribute source recordings.\n"""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport time\nfrom html.parser import HTMLParser\nfrom pathlib import Path\nfrom urllib.parse import urljoin, urlsplit, unquote, quote\n\nimport requests\nfrom lensfd_full_run import inspect_hdf5, hash_file, atomic_json, _load_journal, _json_line, utcnow\n\nDATASET_ROOTS = (\n    \'https://renew.rice.edu/datasets/ashu/traces/LensFD/data_LensFD/\',\n    \'https://renew-wireless.org/datasets/ashu/traces/LensFD/data_LensFD/\',\n)\nSOURCE_TAR = \'https://renew.rice.edu/datasets/ashu/traces/LensFD/data_lensFD.tgz\'\nSESSION_HEADERS = {\'Accept-Encoding\': \'identity\', \'User-Agent\': \'SANAD-LensFD-full-corpus-audit/3.0\'}\n\n\nclass LinkParser(HTMLParser):\n    def __init__(self):\n        super().__init__()\n        self.links = []\n\n    def handle_starttag(self, tag, attrs):\n        if tag.lower() == \'a\':\n            for k, v in attrs:\n                if k.lower() == \'href\' and v is not None:\n                    self.links.append(v)\n\n\ndef crawl_publisher(root: str, session=None, max_dirs=250) -> dict:\n    """Enumerate every published regular file via server directory listings.\n\n    Do not claim full coverage if listing is disabled, truncated, or ambiguous.\n    Do not cross hosts, traverse parent paths, or count HTML index pages as data.\n    """\n    root = root.rstrip(\'/\') + \'/\'\n    p_root = urlsplit(root)\n    if p_root.scheme not in (\'https\',\'http\'): raise ValueError(\'Invalid scheme\')\n    session = session or requests.Session()\n    pending = [root]\n    visited = set()\n    files = {}\n    while pending:\n        url = pending.pop()\n        if url in visited: continue\n        if len(visited) >= max_dirs: raise RuntimeError(\'Too many source subdirectories; refusing incomplete crawl\')\n        visited.add(url)\n        r = session.get(url, timeout=(20,90), headers=SESSION_HEADERS)\n        r.raise_for_status()\n        ctype = r.headers.get(\'Content-Type\', \'\').lower()\n        if \'html\' not in ctype and not r.text.lstrip().lower().startswith((\'<!doctype html\',\'<html\')):\n            raise RuntimeError(f\'Publisher directory index unavailable: {url}, Content-Type={ctype}\')\n        if len(r.content) > 15 * 1024 * 1024:\n            raise RuntimeError(\'Oversized directory index; cannot verify completeness\')\n        links = LinkParser(); links.feed(r.text)\n        good = 0\n        for href in links.links:\n            if href in (\'../\',\'./\',\'#\') or href.startswith((\'../\',\'?\',\'#\',\'mailto:\')):\n                continue\n            resolved = urlsplit(urljoin(url, href))\n            if (resolved.scheme, resolved.netloc) != (p_root.scheme, p_root.netloc):\n                continue\n            if resolved.query or resolved.fragment:\n                continue\n            target = resolved._replace(query=\'\',fragment=\'\').geturl()\n            if not target.startswith(root):\n                continue\n            rel = unquote(target[len(root):])\n            if not rel or rel.startswith(\'/\') or \'/..\' in rel or \'\\\\\' in rel:\n                continue\n            if target.endswith(\'/\'):\n                if target != url and target not in visited:\n                    pending.append(target);good += 1\n                continue\n            if rel in files and files[rel] != target:\n                raise RuntimeError(f\'Ambiguous listing entry: {rel}\')\n            files[rel] = target\n            good += 1\n        if not good and url == root:\n            raise RuntimeError(f\'Directory listing blocked or empty: {url}; no archive re-download attempted\')\n    if not files: raise RuntimeError(\'Publisher returned no data files; cannot assert full coverage\')\n    if not any(Path(k).suffix.lower() in (\'.hdf5\',\'.h5\') for k in files):\n        raise RuntimeError(\'No HDF5 files in direct publisher listing; stop rather than fabricate full scan\')\n    return dict(root=root,directories=len(visited),files=dict(sorted(files.items())))\n\n\ndef discover_all(output_dir, roots=DATASET_ROOTS, session=None):\n    """Try official directory sources. Stop without touching the broken .tgz."""\n    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)\n    attempts=[]\n    for root in roots:\n        try:\n            manifest=crawl_publisher(root,session=session)\n            manifest[\'source_tar_reference\']=SOURCE_TAR\n            manifest[\'file_count\']=len(manifest[\'files\'])\n            manifest[\'hdf5_file_count\']=sum(Path(x).suffix.lower() in (\'.hdf5\',\'.h5\') for x in manifest[\'files\'])\n            manifest[\'discovered_utc\']=utcnow()\n            atomic_json(output_dir/\'DIRECT_SOURCE_MANIFEST.json\',manifest)\n            return manifest\n        except Exception as e:\n            attempts.append({\'root\':root,\'error\':f\'{type(e).__name__}: {e}\'})\n    atomic_json(output_dir/\'DIRECT_DISCOVERY_ERROR.json\',{\'attempts\':attempts,\'status\':\'NO_COMPLETE_SOURCE_INDEX\'})\n    raise RuntimeError(\'No accessible publisher directory listing; see DIRECT_DISCOVERY_ERROR.json. \'\n                       \'Do not re-download a known-bad .tgz.\')\n\n\ndef probe_remote(url,session):\n    """Inspect source size and range support without downloading payload."""\n    try:\n        r=session.head(url,timeout=(15,60),allow_redirects=True,headers=SESSION_HEADERS)\n        if not r.ok: return None,None\n        length=int(r.headers[\'Content-Length\']) if r.headers.get(\'Content-Length\',\'\').isdigit() else None\n        return length,r.headers.get(\'ETag\')\n    except requests.RequestException:\n        return None,None\n\n\ndef download_full_file(url,dest,session=None,max_attempts=8):\n    """Verified size (where advertised), retry with Range; never claim partial as full."""\n    session=session or requests.Session()\n    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True)\n    part=dest.with_name(dest.name+\'.part\')\n    expected,etag=probe_remote(url,session)\n    last=None\n    for attempt in range(1,max_attempts+1):\n        offset=part.stat().st_size if part.exists() else 0\n        if expected is not None and offset>expected:\n            part.unlink();offset=0\n        if expected is not None and offset==expected and offset>0:\n            part.replace(dest)\n            return {\'size\':expected,\'sha256\':hash_file(dest),\'downloaded_from\':url,\'verified_bytes\':expected}\n        head=dict(SESSION_HEADERS)\n        if offset:\n            head[\'Range\']=f\'bytes={offset}-\'\n            if etag and not etag.startswith(\'W/\'):\n                head[\'If-Range\']=etag\n        try:\n            with session.get(url,headers=head,stream=True,timeout=(25,180)) as resp:\n                if resp.status_code in (403,404):\n                    raise FileNotFoundError(f\'Publisher responded {resp.status_code}: {url}\')\n                resp.raise_for_status()\n                if offset:\n                    cr=resp.headers.get(\'Content-Range\',\'\')\n                    if resp.status_code != 206 or not cr.startswith(f\'bytes {offset}-\'):\n                        # Never append a full 200 response to prior partial contents.\n                        part.unlink(missing_ok=True)\n                        last=\'Range not honored; restarting without mixing bytes\'\n                        continue\n                if expected is None:\n                    cr=resp.headers.get(\'Content-Range\',\'\')\n                    if \'/\' in cr and cr.rsplit(\'/\',1)[-1].isdigit():\n                        expected=int(cr.rsplit(\'/\',1)[-1])\n                    elif not offset and resp.headers.get(\'Content-Length\',\'\').isdigit():\n                        expected=int(resp.headers[\'Content-Length\'])\n                mode=\'ab\' if offset else \'wb\'\n                with part.open(mode) as f:\n                    for block in resp.iter_content(8*1024*1024):\n                        if block: f.write(block)\n                n=part.stat().st_size\n                if expected is not None and n>expected:\n                    part.unlink()\n                    raise RuntimeError(f\'Publisher file exceeded claimed size: {n}>{expected}\')\n                if expected is not None and n<expected:\n                    last=f\'Partial transfer {n}/{expected}; retry Range\'\n                    continue\n                if expected is None and not resp.headers.get(\'Content-Length\'):\n                    raise RuntimeError(f\'Cannot verify complete transfer without publisher byte count: {url}\')\n                part.replace(dest)\n                return {\'size\':dest.stat().st_size,\'sha256\':hash_file(dest),\'downloaded_from\':url,\n                        \'verified_bytes\':dest.stat().st_size}\n        except (requests.RequestException, OSError) as exc:\n            last=f\'{type(exc).__name__}: {exc}\'\n            time.sleep(min(attempt,4))\n    raise RuntimeError(f\'Individual-file download failed after {max_attempts} attempts: {url}; {last}\')\n\n\ndef run_all_direct(*,manifest,results,scratch,old_archive_journal=None,\n                   strict=True,revalidate_previous_archive_members=False,session=None):\n    """Process entire direct-file manifest and merge prior successful audits safely."""\n    results=Path(results);results.mkdir(parents=True,exist_ok=True)\n    scratch=Path(scratch);scratch.mkdir(parents=True,exist_ok=True)\n    session=session or requests.Session()\n    old=_load_journal(Path(old_archive_journal)) if old_archive_journal else {}\n    direct_journal=results/\'direct_members.jsonl\'\n    done=_load_journal(direct_journal)\n    files=manifest[\'files\']; errors=[]\n    if len(files)!=len(set(files)):raise RuntimeError(\'Duplicate manifest names\')\n    # In the tar the root was "data_LensFD". Preserve exact relative path mapping.\n    old_by_rel={r.removeprefix(\'data_LensFD/\'):v for r,v in old.items()\n                if r.startswith(\'data_LensFD/\') and v.get(\'status\')==\'OK\'}\n    # A previously verified archive file absent from the publisher listing indicates\n    # the direct listing and archive may not describe the same corpus.\n    missing_old=set(old_by_rel)-set(files)\n    if missing_old:\n        atomic_json(results/\'DIRECT_MANIFEST_MISMATCH.json\',{\'missing_archive_prefix_members\':sorted(missing_old)})\n        raise RuntimeError(f\'Direct index missing {len(missing_old)} previously verified archived files. \'\n                           \'Cannot attest to full publisher corpus.\')\n    for idx,(rel,url) in enumerate(sorted(files.items()),start=1):\n        if rel in done and done[rel].get(\'status\')==\'OK\' and not revalidate_previous_archive_members:\n            continue\n        prev=old_by_rel.get(rel)\n        if prev and not revalidate_previous_archive_members and prev.get(\'sha256\'):\n            size,etag=probe_remote(url,session)\n            if size is not None and size!=prev.get(\'size\'):\n                raise RuntimeError(f\'Source mismatch for previously verified member: {rel}; \'\n                                   f\'direct={size} archive={prev.get("size")}\')\n            row={**prev,\'name\':rel,\'source\':SOURCE_TAR,\'status\':\'OK\',\n                 \'reused_verified_archive_prefix\':True,\n                 \'direct_publisher_size_confirmed\':size is not None,\n                 \'direct_file_sha256_revalidated\':False,\n                 \'explanation\':\'This member was fully scanned and hashed inside readable publisher archive prefix.\'}\n            _json_line(direct_journal,row);done[rel]=row\n            print(f\'[{idx}/{len(files)}] REUSE {rel}\',flush=True)\n            continue\n        row={\'name\':rel,\'url\':url,\'status\':\'ERROR\',\'source\':\'direct_publisher\'}\n        local=scratch/Path(rel).name\n        # Unique work file name even when different directories have same base filename\n        local=scratch/(hashlib.sha256(rel.encode()).hexdigest()[:14]+\'-\'+Path(rel).name)\n        try:\n            # File is downloaded individually, never a subset or a range sample.\n            expected,_=probe_remote(url,session)\n            disk=shutil.disk_usage(scratch).free\n            if expected is not None and disk<expected+256*1024*1024:\n                raise OSError(f\'INSUFFICIENT_SCRATCH_DISK {rel}: required \'\n                              f\'{expected/2**30:.2f} GiB + margin; available {disk/2**30:.2f} GiB. \'\n                              \'Use a larger disk runtime. Never sample.\')\n            downloaded=download_full_file(url,local,session=session)\n            row.update(size=downloaded[\'size\'],sha256=downloaded[\'sha256\'])\n            if prev and revalidate_previous_archive_members and prev.get(\'sha256\')!=row[\'sha256\']:\n                raise RuntimeError(f\'Previously verified archive member differs from direct publisher: {rel}\')\n            if prev and revalidate_previous_archive_members and prev.get(\'sha256\')==row[\'sha256\']:\n                row.update({k:v for k,v in prev.items() if k in (\'hdf5_datasets\',\'hdf5_dataset_count\',\n                          \'numeric_dataset_count\',\'numeric_values_scanned\',\'total_logical_values\',\'root_attribute_names\')})\n                row[\'analysis\']=\'REVALIDATED_SHA_REUSE_HDF5_STATISTICS\'\n            elif Path(rel).suffix.lower() in (\'.hdf5\',\'.h5\'):\n                row.update(inspect_hdf5(local))\n                row[\'analysis\']=\'FULL_HDF5_NUMERIC_SCAN\'\n            else:\n                row[\'analysis\']=\'FULL_FILE_SHA256\'\n            row[\'status\']=\'OK\'\n        except Exception as exc:\n            row[\'error\']=f\'{type(exc).__name__}: {exc}\'\n            errors.append({\'name\':rel,\'error\':row[\'error\']})\n        finally:\n            local.unlink(missing_ok=True)\n        _json_line(direct_journal,row)\n        done[rel]=row\n        print(f\'[{idx}/{len(files)}] {row["status"]} {rel}\',flush=True)\n        # Stop on first failure: avoid expensive partial work on already-bad source.\n        if row[\'status\']!=\'OK\' and strict:break\n    covered=sum(r.get(\'status\')==\'OK\' for rel,r in done.items() if rel in files)\n    full=covered==len(files)\n    summary={\'status\':\'COMPLETE_ALL_PUBLISHED_DIRECT_FILES\' if full else \'INCOMPLETE\',\n             \'official_root\':manifest[\'root\'],\'manifest_count\':len(files),\n             \'manifest_hdf5_count\':manifest[\'hdf5_file_count\'],\n             \'files_ok\':covered,\'files_missing_or_failed\':len(files)-covered,\n             \'numeric_values_scanned\':sum(r.get(\'numeric_values_scanned\',0) for rel,r in done.items() if rel in files),\n             \'reused_verified_archive_prefix\':sum(bool(r.get(\'reused_verified_archive_prefix\')) for rel,r in done.items() if rel in files),\n             \'source_archive_is_known_broken\':True,\n             \'no_sampling\':True,\'errors\':errors, \'finished_utc\':utcnow()}\n    atomic_json(results/\'DIRECT_FULL_DATA_SUMMARY.json\',summary)\n    if strict and not full: raise RuntimeError(\'Full direct LensFD audit INCOMPLETE; see DIRECT_FULL_DATA_SUMMARY.json: \'+str(errors[-1:] if errors else []))\n    return summary\n', encoding="utf-8")
print("Installed lensfd_direct_full.py")

In [ ]:
from pathlib import Path
Path('/content/test_lensfd_direct_full.py').write_text("import hashlib\nimport json\nimport os\nfrom pathlib import Path\nfrom functools import partial\nfrom http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer\nfrom threading import Thread\n\nimport h5py\nimport numpy as np\nimport pytest\n\nfrom lensfd_direct_full import (crawl_publisher,discover_all,run_all_direct,download_full_file)\n\n\nclass Quiet(SimpleHTTPRequestHandler):\n    def log_message(self,*a):pass\n\n@pytest.fixture\ndef host(tmp_path):\n    public=tmp_path/'web';pub=public/'data_LensFD';(pub/'outdoor').mkdir(parents=True)\n    a=pub/'outdoor'/'a.hdf5'\n    with h5py.File(a,'w') as f:\n        f.create_dataset('csi',data=np.ones((17,7,3),dtype=np.complex64))\n        f.create_dataset('iq',data=np.arange(40,dtype=np.float64))\n    with h5py.File(pub/'b.hdf5','w') as f:\n        f.create_dataset('q',data=np.array([1,2,3,4],dtype=int))\n    (pub/'LICENSE').write_text('dataset license')\n    s=ThreadingHTTPServer(('127.0.0.1',0),partial(Quiet,directory=str(public)))\n    t=Thread(target=s.serve_forever,daemon=True);t.start()\n    yield f'http://127.0.0.1:{s.server_port}/data_LensFD/',pub\n    s.shutdown();t.join()\n\n\ndef test_crawl_all_publisher_files(host):\n    root,_=host\n    m=crawl_publisher(root)\n    assert sorted(m['files'])==['LICENSE','b.hdf5','outdoor/a.hdf5']\n    assert m['directories']==2\n\n\ndef test_full_direct_no_sampling(host,tmp_path):\n    root,_=host\n    m=discover_all(tmp_path/'results',roots=[root])\n    assert m['hdf5_file_count']==2\n    d=run_all_direct(manifest=m,results=tmp_path/'results',scratch=tmp_path/'scratch')\n    assert d['status']=='COMPLETE_ALL_PUBLISHED_DIRECT_FILES'\n    assert d['files_ok']==3\n    assert d['numeric_values_scanned']==17*7*3+40+4\n    again=run_all_direct(manifest=m,results=tmp_path/'results',scratch=tmp_path/'scratch')\n    assert again['files_ok']==3\n    assert len((tmp_path/'results'/'direct_members.jsonl').read_text().splitlines())==3\n\n\ndef test_reuse_archive_112_style(host,tmp_path):\n    root,pub=host\n    m=discover_all(tmp_path/'results',roots=[root])\n    p=pub/'outdoor'/'a.hdf5'\n    from lensfd_full_run import inspect_hdf5\n    old={**inspect_hdf5(p),'name':'data_LensFD/outdoor/a.hdf5','sha256':hashlib.sha256(p.read_bytes()).hexdigest(),\n         'size':p.stat().st_size,'status':'OK'}\n    journal=tmp_path/'prior_members.jsonl';journal.write_text(json.dumps(old)+'\\n')\n    d=run_all_direct(manifest=m,results=tmp_path/'results',scratch=tmp_path/'scratch',old_archive_journal=journal)\n    assert d['status']=='COMPLETE_ALL_PUBLISHED_DIRECT_FILES'\n    assert d['reused_verified_archive_prefix']==1\n    assert d['numeric_values_scanned']==17*7*3+40+4\n\n\ndef test_incomplete_listing_not_claimed_complete(host,tmp_path):\n    root,pub=host\n    m=discover_all(tmp_path/'results',roots=[root])\n    # Record a prior artifact not present on directory listing: cross-source mismatch.\n    journal=tmp_path/'old.jsonl'\n    journal.write_text(json.dumps({'name':'data_LensFD/nonexistent.hdf5','status':'OK','sha256':'a'*64,'size':100})+'\\n')\n    with pytest.raises(RuntimeError,match='missing'):\n        run_all_direct(manifest=m,results=tmp_path/'results',scratch=tmp_path/'scratch',old_archive_journal=journal)\n\n\ndef test_missing_hdf5_refused(tmp_path):\n    pub=tmp_path/'web'/'data_LensFD';pub.mkdir(parents=True)\n    (pub/'LICENSE').write_text('hi')\n    s=ThreadingHTTPServer(('127.0.0.1',0),partial(Quiet,directory=str(tmp_path/'web')))\n    t=Thread(target=s.serve_forever,daemon=True);t.start()\n    try:\n        with pytest.raises(RuntimeError,match='No HDF5'):\n            crawl_publisher(f'http://127.0.0.1:{s.server_port}/data_LensFD/')\n    finally:\n        s.shutdown();t.join()\n", encoding="utf-8")
print("Installed test_lensfd_direct_full.py")

## Step 1 — Local regression tests (small synthetic fixture only)

These tests verify code paths, not publisher data. Real-data processing begins in Step 2 and covers the **entire** publisher index.


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pytest','-q','/content/test_lensfd_direct_full.py'],cwd='/content',check=True)


## Step 2 — Enumerate every official source file before downloading

This interrogates the actual publisher directory listing and saves `DIRECT_SOURCE_MANIFEST.json`. If the publisher blocks listing or direct access, the notebook reports the problem **without downloading the defective `.tgz` again**.


In [ ]:
import sys
sys.path.insert(0,'/content')
from lensfd_direct_full import discover_all,run_all_direct,probe_remote
import requests
MANIFEST=discover_all(RESULTS)
print('Official source:',MANIFEST['root'])
print('Total listed regular files:',MANIFEST['file_count'])
print('Total listed HDF5 files:',MANIFEST['hdf5_file_count'])
print('Directory indexes traversed:',MANIFEST['directories'])
for key in [k for k in MANIFEST['files'] if k.endswith('no-s2n.hdf5')]:
    with requests.Session() as s:
        n,etag=probe_remote(MANIFEST['files'][key],s)
    print('Previously blocked 20.11-GiB raw file:',key,'Publisher HEAD bytes:',n)
    if n is not None and n != 21589436536:
        print('WARNING: Direct source size differs from the old archive member. Inspect before proceeding.')
assert MANIFEST['file_count']>=1 and MANIFEST['hdf5_file_count']>=1


## Step 3 — Fully scan every published file (zero sampling)

One file at a time: direct HTTP download, exact content-length verification, SHA-256, full HDF5 numeric scan in bounded RAM, checkpoint, delete temporary file. The 112+ `OK` archive members are reused by default with explicit provenance. If the publisher directory lists an original file as missing, this **fails** rather than claiming full coverage. The largest known file needs at least **21 GiB free local scratch**.


In [ ]:
FULL_DIRECT=run_all_direct(manifest=MANIFEST,results=RESULTS,scratch=SCRATCH,
    old_archive_journal=OLD_JOURNAL if OLD_JOURNAL.exists() else None,
    revalidate_previous_archive_members=REVALIDATE_OLD_FILES,
    strict=True)
print(json.dumps(FULL_DIRECT,indent=2))
assert FULL_DIRECT['files_ok']==FULL_DIRECT['manifest_count']
assert FULL_DIRECT['status']=='COMPLETE_ALL_PUBLISHED_DIRECT_FILES'
print('FULL SOURCE COMPLETE: ',FULL_DIRECT['files_ok'],'of',FULL_DIRECT['manifest_count'])


## Step 4 — Verify frozen RC3 has not changed

This verifies the original model and research test lock. It **does not** invent native 7-array × 300-report inputs from LensFD recordings.


In [ ]:
FROZEN_REF='frozen/rc3-sionna-qualified-2026-11-14'
FROZEN_COMMIT='182e77b485cceb770ee6570ba332205c73823c91'
ENGINE_DIR=Path('/content/SANAD-Network-RC3-FROZEN')
if not ENGINE_DIR.exists():
    subprocess.run(['git','clone','--depth','1','--branch',FROZEN_REF,
        'https://github.com/majdishokri/SANAD-Network-RC3.git',str(ENGINE_DIR)],check=True)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ENGINE_DIR,text=True).strip()
assert commit==FROZEN_COMMIT, f'Frozen RC3 commit changed: {commit}'
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],cwd=ENGINE_DIR,check=True)
subprocess.run([sys.executable,'verify_freeze.py'],cwd=ENGINE_DIR,check=True)
subprocess.run([sys.executable,'qualification/verify_qualification.py'],cwd=ENGINE_DIR,check=True)
print('RC3 frozen reference verified:',commit)


In [ ]:
FINAL={
    'dataset':'Rice LensFD, full publisher direct index',
    'status':FULL_DIRECT['status'],
    'publisher_root':MANIFEST['root'],
    'published_regular_files':FULL_DIRECT['manifest_count'],
    'files_processed':FULL_DIRECT['files_ok'],
    'hdf5_numeric_values_scanned':FULL_DIRECT['numeric_values_scanned'],
    'reused_verified_original_tar_prefix_members':FULL_DIRECT['reused_verified_archive_prefix'],
    'revalidate_old_files':REVALIDATE_OLD_FILES,
    'no_sampling':True,
    'frozen_rc3_commit':commit,
    'frozen_rc3_threshold':.50,
    'note':'Recorded RF-data full ingest; does not fabricate RC3 7-array / 300-report observations.'
}
(RESULTS/'RUN_FINAL_STATUS.json').write_text(json.dumps(FINAL,indent=2))
print(json.dumps(FINAL,indent=2))
print('Results:',RESULTS)


**Result files in Drive:** `results_direct/DIRECT_SOURCE_MANIFEST.json`, `direct_members.jsonl`, `DIRECT_FULL_DATA_SUMMARY.json`, `RUN_FINAL_STATUS.json`. The original `results_full/members.jsonl` is never modified.

If direct file listing is denied, send the `DIRECT_DISCOVERY_ERROR.json` details to the RENEW maintainers and request a repaired `.tgz` or a complete file manifest with per-file URLs. Do **not** treat a truncated publisher source as a full dataset.